# Lab 1B · Your first CNN

**Computer Vision for Developers** · Day 1 · Block 5 · Acts 4–7 · by **Musa Ibn Rashid**, SDAIA Academy

**Goal:** Train an MLP baseline and a small CNN on CIFAR-10, trace shapes layer by layer, add augmentation and dropout, read the curves and the confusion matrix, then compare with a pretrained ResNet-18.

## How to use this notebook

1. Switch the runtime to a **T4 GPU** first: *Runtime → Change runtime type → T4 GPU*.
2. Run the cells **from top to bottom** with *Shift + Enter*. Do not skip cells: later cells use variables from earlier ones.
3. Cells with a **TODO** comment need **one line** from you. The cell right after each TODO is a check: it prints ✅ when your line is right, or a hint when it is not.
4. Nobody writes a cell from empty. If you are stuck for more than five minutes, open the *solved* version of this notebook and compare.
5. Colab forgets files when the session ends. If you come back later, run everything again from the top.
6. The **Extra tasks** at the end are optional and are not graded.

Expected time: about 30 minutes in class; finish the rest at home.

In [ ]:
# GPU check, with a CPU fallback
import os, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found. Runtime → Change runtime type → T4 GPU, then Runtime → Restart.")
    print("   You can keep going on CPU: the notebook switches to a 10,000-image subset and fewer epochs.")
# FAST mode shrinks data and epochs so everything still finishes on a CPU.
FAST = DEVICE == "cpu" or os.environ.get("CVD_FAST") == "1"
print("DEVICE =", DEVICE, "| FAST =", FAST)

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
torch.manual_seed(0); np.random.seed(0)
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

## 4 · Learning from data

### 4.1 · The data: CIFAR-10
50,000 training and 10,000 test images, 32 × 32 colour, 10 classes. `ToTensor()` changes the layout from (H, W, C) to **(C, H, W)** *and* divides by 255; `Normalize` then centres each channel.

In [ ]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
normalize = transforms.Normalize(MEAN, STD)
plain_tf = transforms.Compose([transforms.ToTensor(), normalize])

train_full = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=plain_tf)
test_full = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=plain_tf)
CLASSES = train_full.classes
n_train, n_test = (10000, 2000) if FAST else (len(train_full), len(test_full))
train_set = Subset(train_full, range(n_train))
test_set = Subset(test_full, range(n_test))
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, num_workers=2)
x, y = next(iter(train_loader))
print("batch:", tuple(x.shape), "labels:", tuple(y.shape), "| train", n_train, "test", n_test)
print("iterations per epoch:", len(train_loader))

In [ ]:
imgs = x[:16] * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, im, lab in zip(axes.ravel(), imgs, y[:16]):
    ax.imshow(im.permute(1, 2, 0).clamp(0, 1)); ax.set_title(CLASSES[lab], fontsize=9); ax.axis("off")
plt.show()

### 4.2 · Training and evaluation helpers
This is the five-line training loop from slide 1.90. Read it line by line.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()                                   # dropout on, BatchNorm uses the batch
    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)                     # forward
        loss = loss_fn(outputs, labels)             # how wrong
        pass  # TODO: clear old gradients, compute gradients, update weights (three calls on one line, separated by ;)
        loss_sum += loss.item() * len(labels)
        correct += (outputs.argmax(1) == labels).sum().item(); total += len(labels)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()                                    # dropout off, BatchNorm uses saved statistics
    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss_sum += loss_fn(outputs, labels).item() * len(labels)
        correct += (outputs.argmax(1) == labels).sum().item(); total += len(labels)
    return loss_sum / total, correct / total

def fit(model, epochs, lr=1e-3, loader=None):
    loader = loader or train_loader
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for ep in range(epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, loader, opt)
        vl, va = evaluate(model, test_loader)
        for k, v in zip(hist, (tl, vl, ta, va)): hist[k].append(v)
        print(f"epoch {ep+1}/{epochs}  train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f} | {time.time()-t0:.0f}s")
    return hist

In [ ]:
# Check
from torch.utils.data import TensorDataset
_m = nn.Sequential(nn.Flatten(), nn.Linear(4, 2)).to(DEVICE)
_before = _m[1].weight.detach().clone()
_dl = DataLoader(TensorDataset(torch.randn(8, 1, 2, 2), torch.randint(0, 2, (8,))), batch_size=8)
train_one_epoch(_m, _dl, torch.optim.SGD(_m.parameters(), lr=0.5))
try:
    assert not torch.equal(_before, _m[1].weight.detach())
    print('✅ One training step changed the weights: your loop updates the model.')
except Exception as e:
    print('❌ Not yet. Hint: optimizer.zero_grad(); loss.backward(); optimizer.step()')

### 4.3 · The obvious attempt: an MLP
Flatten the 3 × 32 × 32 image into 3,072 numbers and use fully connected layers (3,072 → 512 → 256 → 10).

In [ ]:
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3 * 32 * 32, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 10),
).to(DEVICE)
n_params = 0  # TODO: count the parameters: sum of p.numel() over mlp.parameters()
print(f"MLP parameters: {n_params:,}")

In [ ]:
# Check
try:
    assert n_params == 1_707_274
    print('✅ MLP has 1,707,274 parameters (≈ 1.7 M, slide 1.82).')
except Exception as e:
    print('❌ Not yet. Hint: sum(p.numel() for p in mlp.parameters())')

In [ ]:
mlp_hist = fit(mlp, epochs=2 if FAST else 5)

In [ ]:
# Escape hatch: load our trained weights instead of training
# Use this if the training cell above was too slow or crashed. It downloads weights we trained with exactly this code.
USE_PRETRAINED = False   # ← set to True and run this cell
if USE_PRETRAINED:
    url = "https://raw.githubusercontent.com/mibnrashid/SDAIA_ComputerVision/main/notebooks/weights/mlp_cifar10.pt"
    mlp.load_state_dict(torch.hub.load_state_dict_from_url(url, map_location=DEVICE, file_name="mlp_cifar10.pt"))
    mlp.to(DEVICE)
    print("Loaded mlp_cifar10.pt. Test accuracy:", round(evaluate(mlp, test_loader)[1], 3))

## 5 · The convolutional neural network

The model from slide 1.81: three conv blocks (conv 3×3, padding 1, ReLU, max-pool 2) then a small classifier head.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.head(self.features(x))

cnn = SmallCNN().to(DEVICE)
print(f"CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}")

### 5.1 · Trace the shapes, layer by layer
Use the output-size formula ⌊(W − K + 2P)/S⌋ + 1. Predict each line before you run it.

In [ ]:
x = torch.zeros(1, 3, 32, 32, device=DEVICE)
print(f"{'input':<28}{tuple(x.shape)}")
for layer in list(cnn.features) + list(cnn.head):
    pass  # TODO: pass x through this layer (x = layer(x))
    print(f"{layer.__class__.__name__:<28}{tuple(x.shape)}")
final_shape = tuple(x.shape)

In [ ]:
# Check
try:
    assert final_shape == (1, 10)
    print('✅ Final output is (1, 10): ten class scores.')
except Exception as e:
    print('❌ Not yet. Hint: inside the loop write x = layer(x)')

In [ ]:
cnn_hist = fit(cnn, epochs=2 if FAST else 8)

In [ ]:
# Escape hatch: load our trained weights instead of training
# Use this if the training cell above was too slow or crashed. It downloads weights we trained with exactly this code.
USE_PRETRAINED = False   # ← set to True and run this cell
if USE_PRETRAINED:
    url = "https://raw.githubusercontent.com/mibnrashid/SDAIA_ComputerVision/main/notebooks/weights/cnn_cifar10.pt"
    cnn.load_state_dict(torch.hub.load_state_dict_from_url(url, map_location=DEVICE, file_name="cnn_cifar10.pt"))
    cnn.to(DEVICE)
    print("Loaded cnn_cifar10.pt. Test accuracy:", round(evaluate(cnn, test_loader)[1], 3))

## 6 · Training it properly

### 6.1 · Reading the curves
Always plot **both** curves. Train loss falling while val loss rises = overfitting.

In [ ]:
def plot_curves(hists, names):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for h, n in zip(hists, names):
        axes[0].plot(h["train_loss"], "--", label=f"{n} train"); axes[0].plot(h["val_loss"], label=f"{n} val")
        axes[1].plot(h["val_acc"], label=n)
    axes[0].set_title("loss (dashed = train)"); axes[1].set_title("validation accuracy")
    for a in axes: a.set_xlabel("epoch"); a.legend(fontsize=8)
    plt.show()
plot_curves([mlp_hist, cnn_hist], ["MLP", "CNN"])

### 6.2 · Augmentation + dropout
Random crops and flips give a slightly different image every epoch; dropout switches off half the head's neurons during training. Both fight overfitting.

In [ ]:
train_tf = plain_tf  # TODO: Compose([RandomCrop(32, padding=4), RandomHorizontalFlip(), ToTensor(), normalize])
aug_train = Subset(torchvision.datasets.CIFAR10("data", train=True, download=False, transform=train_tf), range(n_train))
aug_loader = DataLoader(aug_train, batch_size=128, shuffle=True, num_workers=2)
print(train_tf)

In [ ]:
# Check
try:
    assert any(isinstance(t, transforms.RandomHorizontalFlip) for t in train_tf.transforms) and any(isinstance(t, transforms.RandomCrop) for t in train_tf.transforms)
    print('✅ Augmentation pipeline has RandomCrop and RandomHorizontalFlip.')
except Exception as e:
    print('❌ Not yet. Hint: add RandomCrop(32, padding=4) and RandomHorizontalFlip() before ToTensor().')

In [ ]:
cnn_aug = SmallCNN(dropout=0.5).to(DEVICE)
aug_hist = fit(cnn_aug, epochs=2 if FAST else 12, loader=aug_loader)
plot_curves([cnn_hist, aug_hist], ["CNN", "CNN + aug + dropout"])

In [ ]:
# Escape hatch: load our trained weights instead of training
# Use this if the training cell above was too slow or crashed. It downloads weights we trained with exactly this code.
USE_PRETRAINED = False   # ← set to True and run this cell
if USE_PRETRAINED:
    url = "https://raw.githubusercontent.com/mibnrashid/SDAIA_ComputerVision/main/notebooks/weights/cnn_aug_cifar10.pt"
    cnn_aug.load_state_dict(torch.hub.load_state_dict_from_url(url, map_location=DEVICE, file_name="cnn_aug_cifar10.pt"))
    cnn_aug.to(DEVICE)
    print("Loaded cnn_aug_cifar10.pt. Test accuracy:", round(evaluate(cnn_aug, test_loader)[1], 3))

### 6.3 · The confusion matrix
Rows = true class, columns = predicted class. The diagonal is correct; the off-diagonal shows **which** classes get confused.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

@torch.no_grad()
def predict_all(model, loader):
    model.eval(); ys, ps = [], []
    for images, labels in loader:
        ps.append(model(images.to(DEVICE)).argmax(1).cpu()); ys.append(labels)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

y_true, y_pred = predict_all(cnn_aug, test_loader)
cm = None  # TODO: build the confusion matrix with sklearn's confusion_matrix(y_true, y_pred)
if cm is not None:
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(xticks_rotation=45, cmap="Blues", colorbar=False)
    plt.gcf().set_size_inches(7, 6); plt.show()

In [ ]:
# Check
try:
    assert cm.shape == (10, 10) and cm.sum() == len(y_true)
    print('✅ Confusion matrix is 10 × 10 and covers every test image.')
except Exception as e:
    print('❌ Not yet. Hint: confusion_matrix(y_true, y_pred)')

In [ ]:
per_class = cm.diagonal() / cm.sum(1)
for c, a in sorted(zip(CLASSES, per_class), key=lambda t: t[1]):
    print(f"{c:<12}{a:.2f}")
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"\nMost common mistake: true {CLASSES[i]} predicted as {CLASSES[j]} ({off[i, j]} times)")

## 7 · Standing on giants: transfer learning

ResNet-18 was trained on 1.2 million ImageNet photos. We keep its backbone, replace the head with a 10-class layer, and fine-tune briefly. Images are upscaled to 96 × 96 so the pretrained filters see objects at a familiar size.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
IMNET_MEAN, IMNET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
res_tf = transforms.Compose([transforms.Resize(96), transforms.RandomHorizontalFlip(), transforms.ToTensor(),
                             transforms.Normalize(IMNET_MEAN, IMNET_STD)])
res_test_tf = transforms.Compose([transforms.Resize(96), transforms.ToTensor(), transforms.Normalize(IMNET_MEAN, IMNET_STD)])
n_res = 5000 if FAST else 20000
res_train = Subset(torchvision.datasets.CIFAR10("data", train=True, transform=res_tf), range(n_res))
res_test = Subset(torchvision.datasets.CIFAR10("data", train=False, transform=res_test_tf), range(n_test))
res_loader = DataLoader(res_train, batch_size=128, shuffle=True, num_workers=2)
res_test_loader = DataLoader(res_test, batch_size=256, num_workers=2)

resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
resnet.fc = nn.Linear(resnet.fc.in_features, 10)      # new head: 512 → 10
resnet = resnet.to(DEVICE)
print("new head:", resnet.fc)

In [ ]:
opt = torch.optim.Adam(resnet.parameters(), lr=3e-4)
for ep in range(1 if FAST else 2):
    t0 = time.time()
    tl, ta = train_one_epoch(resnet, res_loader, opt)
    vl, va = evaluate(resnet, res_test_loader)
    print(f"epoch {ep+1}  train acc {ta:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")
res_acc = va

In [ ]:
# Escape hatch: if fine-tuning is too slow, freeze the backbone and train only the head for 1 epoch on 5,000 images
FREEZE_INSTEAD = False   # ← set to True and run this cell
if FREEZE_INSTEAD:
    for p in resnet.parameters(): p.requires_grad = False
    for p in resnet.fc.parameters(): p.requires_grad = True
    small = DataLoader(Subset(res_train, range(5000)), batch_size=128, shuffle=True, num_workers=2)
    tl, ta = train_one_epoch(resnet, small, torch.optim.Adam(resnet.fc.parameters(), lr=1e-3))
    res_acc = evaluate(resnet, res_test_loader)[1]
    print("frozen backbone, head only: val acc", round(res_acc, 3))

In [ ]:
results = {"MLP": mlp_hist["val_acc"][-1], "Small CNN": cnn_hist["val_acc"][-1],
           "CNN + aug + dropout": aug_hist["val_acc"][-1], "ResNet-18 (pretrained)": res_acc}
for k, v in results.items():
    print(f"{k:<26}{v:.3f}  " + "█" * int(v * 40))

### Tonight's deliverable
Write **one paragraph** in the cell below comparing the MLP, the CNN and the pretrained ResNet-18: the accuracy of each, and one observation from the confusion matrix (which classes get confused, and why you think so).

*Your paragraph here.*

## Extra tasks (optional)

1. **Explode the learning rate:** train a fresh SmallCNN for 2 epochs with `lr=1.0`. What happens to the loss?
2. **Shuffle the pixels:** apply the same fixed random permutation to every image's pixels and retrain the MLP. Does its accuracy change? (Slide 1.64.)
3. **Forget `model.eval()`:** evaluate `cnn_aug` twice in train mode. Why do the two numbers differ?
4. **Count by formula:** compute the CNN's parameter count by hand with (k·k·C_in + 1)·C_out and check it equals 620,362.

In [ ]:
# Extra 1 — your code here (hint: fit(SmallCNN().to(DEVICE), epochs=2, lr=1.0))

In [ ]:
# Extra 2 — your code here (hint: a fixed torch.randperm(3072) applied after flattening)

In [ ]:
# Extra 3 — your code here (hint: call model.train() and evaluate the same model twice)

In [ ]:
# Extra 4 — your code here (hint: (k*k*C_in + 1) * C_out for each conv, plus the two Linear layers)